In [4]:
import os

from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

from langchain_community.vectorstores import Chroma

##utility imports
import numpy as np
from typing import List


/var/folders/65/05__y6lj0zxdj8bj5b2kp7jh0000gn/T/ipykernel_12753/2478958902.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/Users/rutwikbelsare/Rutwik/Gen AI/Projects/RagProject/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs

['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [7]:
import tempfile
temp_dir = tempfile.mkdtemp()
for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample Document create in: {temp_dir}")

Sample Document create in: /var/folders/65/05__y6lj0zxdj8bj5b2kp7jh0000gn/T/tmppffykm6x


In [8]:
import tempfile
temp_dir = tempfile.mkdtemp()
for i, doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)


### 2. Document Loading


In [9]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
loader = DirectoryLoader(
    "data",
    glob = "*.txt",
    loader_cls = TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)
documents = loader.load()
print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200]+"...")



Loaded 3 documents

First document preview:

    Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity r...


### Document Splitting

In [10]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500,
                                               chunk_overlap = 50,
                                               length_function = len,
                                               separators = [" "]
                                               )

In [11]:
chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\n Chunk example:")
print(f"Content:{chunks[0].page_content[:150]} ...")
print(f"Metdata: {chunks[0].metadata}")


Created 5 chunks from 3 documents

 Chunk example:
Content:Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NL ...
Metdata: {'source': 'data/doc_2.txt'}


In [12]:
chunks

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervise

### Embedding Models


In [13]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sample_text = "ML is Most facinating."
vectors = model.encode(sample_text)
vectors

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5399.15it/s]


array([ 6.09488785e-02, -8.77699628e-02,  2.79520880e-02, -3.22750653e-03,
       -2.16445476e-02, -4.12241183e-02,  7.92362913e-03,  3.24310139e-02,
        3.11122090e-02, -3.44035104e-02,  2.21525151e-02,  1.88259501e-02,
        6.65721437e-03, -4.29913960e-02, -1.25912696e-01, -2.16882173e-02,
        1.31817043e-01, -1.78541970e-02, -6.43605292e-02, -4.58539315e-02,
       -7.22258016e-02, -2.06151120e-02,  6.78766379e-03,  1.01029962e-01,
        5.68706989e-02,  5.57558462e-02,  2.33796686e-02, -9.38938651e-03,
        2.58984901e-02, -6.53310269e-02, -2.51106378e-02,  4.76905964e-02,
        6.35422245e-02, -2.70509832e-02, -4.07056622e-02, -5.74543420e-03,
        4.97437678e-02, -2.31769364e-02,  2.21755775e-03,  2.80581340e-02,
        2.35335436e-02, -8.15321952e-02,  7.83854350e-02, -3.75521146e-02,
        7.32457591e-03,  5.06329387e-02, -1.30604045e-03, -5.06869070e-02,
        8.25176667e-03,  1.00466341e-01, -6.01508096e-02, -6.04801476e-02,
       -8.16401169e-02,  

## Initialize the chromaDB Vector and store the chunks in Vector Representation

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings


## creata a Chromdb vector store 
persist_directory = "./chroma_db"
embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)
##initilaize the chromadab with hugging face embeddings

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory= persist_directory,
    collection_name = "rag_collection"

)
print(f"Vectors store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to : {persist_directory}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15947.34it/s]


Vectors store created with 20 vectors
Persisted to : ./chroma_db


### test the similarity search

In [15]:
query = "what are the types of Machin learning?"
similar_docs = vectorstore.similarity_search(query, k = 3)
similar_docs

[Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, whi

In [16]:
query = "what is NLP?"
similar_docs = vectorstore.similarity_search(query, k = 3)
similar_docs

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mecha

In [17]:
print(f"Query : {query}")
print(f"\nTop {len(similar_docs)} similar chunks")
for i, doc in enumerate(similar_docs):
    print(f"\n --- Chunk {i+1} ---")
    print(doc.page_content[:200] + "....")
    print(f"Source:{doc.metadata.get('source','Unknown')}")

Query : what is NLP?

Top 3 similar chunks

 --- Chunk 1 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn....
Source:data/doc_2.txt

 --- Chunk 2 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn....
Source:data/doc_2.txt

 --- Chunk 3 ---
Natural Language Processing (NLP)

    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recogn....
Source:data/doc_2.txt


### Advanced similarity search with Scores

In [18]:
results_scores = vectorstore.similarity_search_with_score(query, k = 3)
results_scores

[(Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
  0.6382687091827393),
 (Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These m

In [19]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "Qwen/Qwen2.5-1.5B-Instruct",
    model_provider="huggingface",
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 256,
        "do_sample": False,
        "return_full_text": False,
    },
)

response = llm.invoke("What does LLM stand for in artificial intelligence?")
print(response.content)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 6123.42it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces be

LLM stands for Large Language Model. It refers to a type of machine learning model that is designed to understand and generate human-like language based on vast amounts of text data. These models can be used for various tasks such as natural language processing (NLP), question answering, translation, and more.


## Moden RAG Chain

In [20]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate

from langchain_classic.chains.combine_documents import create_stuff_documents_chain


In [21]:


## Convert vector store to retriever
retriever = vectorstore. as_retriever(
    search_kwarfs = {'k':3} ##  retriever top 3 relevant chunks
)
retriever


VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x128d8e270>, search_kwargs={})

In [22]:
##creeating a prompt template

system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])




In [23]:
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [24]:
### create a document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.2', 'langchain-huggingface': '1.2.3'}}, llm=

In [25]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't know the answer, just say that you don't know. \nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.2', 'langchain-huggingface': '1.2.3'}}, llm=

In [26]:
from langchain_classic.chains import create_retrieval_chain
rag_chain = create_retrieval_chain(retriever, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x128d8e270>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. \nUse the following pieces of retrieved context to answer the question. \nIf you don't 

In [27]:
rag_chain.invoke({"input":"What is Deep Learning?"})

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'input': 'What is Deep Learning?',
 'context': [Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers'),
  Document(metadata={'source': 'data/doc_1.txt'}, page_content='Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processi

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [28]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel


In [29]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x128d8e270>, search_kwargs={})

In [30]:
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")

custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])

In [31]:
##Format the ouput documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [32]:
##Build the chain Using LCEL

rag_chain_lcel = (
    {"context":retriever | format_docs, "question":RunnablePassthrough()}
    | custom_prompt
    |llm
    |StrOutputParser()
)
rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x128d8e270>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following context to answer the question. \nIf you don't know the answer based on the context, say you don't know.\nProvide specific details from the context to support your answer.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"), additional_kwargs={})])
| ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.2', 'langchain-huggingface': '1.2.3'}}, llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': 

In [36]:
rag_chain_lcel=(
    { 
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

In [37]:
response = rag_chain_lcel.invoke("What is Deep learning?")
response

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Deep learning is a subset of machine learning that uses artificial neural networks inspired by the structure and function of the human brain. These networks consist of multiple layers of interconnected nodes or neurons. Deep learning has significantly advanced various domains such as computer vision, natural language processing, and speech recognition through its ability to process complex data structures effectively. Key types include Convolutional Neural Networks (for image processing), Recurrent Neural Networks (for sequence modeling in NLP and time series analysis), and Transformers (which excel in tasks involving long-range dependencies).'

In [49]:
def query_rag_lcel(question):
    # Generate the answer using your LCEL RAG chain
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Retrieve relevant documents
    docs = retriever.invoke(question)

    print("\nSource Documents:")
    for doc in docs:
        print(f"Content: {doc.page_content}")
        print(f"Metadata: {doc.metadata}")
        print("-" * 50)

    return answer

In [50]:
print("Testing LCEL Chain")
query_rag_lcel("What is the key concept in Reinforcement Learning?")

Testing LCEL Chain


[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: The key concept in Reinforcement Learning is that it involves an agent making decisions by interacting with an environment. This process allows the agent to learn how to act optimally in order to maximize cumulative reward over time. Specifically, reinforcement learning algorithms use state-action pairs to determine the best action to take at each step, taking into account both immediate rewards and long-term consequences. The goal is for the agent to develop a policy that maximizes expected future rewards.

Source Documents:
Content: Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient 

'The key concept in Reinforcement Learning is that it involves an agent making decisions by interacting with an environment. This process allows the agent to learn how to act optimally in order to maximize cumulative reward over time. Specifically, reinforcement learning algorithms use state-action pairs to determine the best action to take at each step, taking into account both immediate rewards and long-term consequences. The goal is for the agent to develop a policy that maximizes expected future rewards.'

In [40]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sample_text = "ML is Most facinating."
vectors = model.encode(sample_text)
vectors

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13335.39it/s]


array([ 6.09488785e-02, -8.77699628e-02,  2.79520880e-02, -3.22750653e-03,
       -2.16445476e-02, -4.12241183e-02,  7.92362913e-03,  3.24310139e-02,
        3.11122090e-02, -3.44035104e-02,  2.21525151e-02,  1.88259501e-02,
        6.65721437e-03, -4.29913960e-02, -1.25912696e-01, -2.16882173e-02,
        1.31817043e-01, -1.78541970e-02, -6.43605292e-02, -4.58539315e-02,
       -7.22258016e-02, -2.06151120e-02,  6.78766379e-03,  1.01029962e-01,
        5.68706989e-02,  5.57558462e-02,  2.33796686e-02, -9.38938651e-03,
        2.58984901e-02, -6.53310269e-02, -2.51106378e-02,  4.76905964e-02,
        6.35422245e-02, -2.70509832e-02, -4.07056622e-02, -5.74543420e-03,
        4.97437678e-02, -2.31769364e-02,  2.21755775e-03,  2.80581340e-02,
        2.35335436e-02, -8.15321952e-02,  7.83854350e-02, -3.75521146e-02,
        7.32457591e-03,  5.06329387e-02, -1.30604045e-03, -5.06869070e-02,
        8.25176667e-03,  1.00466341e-01, -6.01508096e-02, -6.04801476e-02,
       -8.16401169e-02,  

## Add New documents to existing vector store

In [41]:
vectorstore

In [42]:
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [43]:
chunks

[Document(metadata={'source': 'data/doc_2.txt'}, page_content='Natural Language Processing (NLP)\n\n    NLP is a field of AI that focuses on the interaction between computers and human language. \n    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, \n    machine translation, and question answering. Modern NLP heavily relies on transformer \n    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand \n    context and relationships between words in text.'),
 Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervise

In [44]:
new_doc = Document(
    page_content=new_document,
    metadata= {
        "source":"manual_Addition",
        "topic":"reinforcement_Learning"
    }
)
new_doc

Document(metadata={'source': 'manual_Addition', 'topic': 'reinforcement_Learning'}, page_content='\nReinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.\n')

In [45]:
###split the doucment
new_chunks = text_splitter.split_documents([new_doc])
new_chunks

[Document(metadata={'source': 'manual_Addition', 'topic': 'reinforcement_Learning'}, page_content='Reinforcement Learning in Detail\n\nReinforcement learning (RL) is a type of machine learning where an agent learns to make \ndecisions by interacting with an environment. The agent receives rewards or penalties \nbased on its actions and learns to maximize cumulative reward over time. Key concepts \nin RL include: states, actions, rewards, policies, and value functions. Popular RL \nalgorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and \nActor-Critic methods. RL has been'),
 Document(metadata={'source': 'manual_Addition', 'topic': 'reinforcement_Learning'}, page_content='methods, and \nActor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), \nrobotics, and autonomous systems.')]

In [46]:
vectorstore.add_documents(new_chunks)

['9a8e0d7d-75d7-4be4-9f52-1d36dc64e82e',
 '49dc9c49-9534-4564-a65d-99d513a21bc3']

In [47]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total Vector store {vectorstore._collection.count()}")


Added 2 new chunks to the vector store
Total Vector store 22


In [51]:
new_question = "What are the keys concepts in reinforcement learning"
result = query_rag_lcel(new_question)
result

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Answer: The key concepts in reinforcement learning are:

1. States: These represent the current situation or condition of the environment that the agent is in. They can be internal states like sensor readings or external states like weather conditions.

2. Actions: These are the choices or decisions made by the agent to interact with the environment. Each action leads to a new state and possibly a reward or penalty.

3. Rewards: These are feedback signals given by the environment after each action. Positive rewards encourage good behavior, while negative rewards discourage it.

4. Policies: A policy defines how the agent should behave in different situations. It maps states to actions.

5. Value Functions: These estimate the expected future rewards for being in a particular state. They help agents decide which actions to take to maximize their total reward.

6. Learning Algorithms: Techniques used to update these models and improve the agent's performance over time. Common algorithms i

"The key concepts in reinforcement learning are:\n\n1. States: These represent the current situation or condition of the environment that the agent is in. They can be internal states like sensor readings or external states like weather conditions.\n\n2. Actions: These are the choices or decisions made by the agent to interact with the environment. Each action leads to a new state and possibly a reward or penalty.\n\n3. Rewards: These are feedback signals given by the environment after each action. Positive rewards encourage good behavior, while negative rewards discourage it.\n\n4. Policies: A policy defines how the agent should behave in different situations. It maps states to actions.\n\n5. Value Functions: These estimate the expected future rewards for being in a particular state. They help agents decide which actions to take to maximize their total reward.\n\n6. Learning Algorithms: Techniques used to update these models and improve the agent's performance over time. Common algorit

## Advanced RAG Techniques- Conversational Memory

In [52]:
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import AIMessage, HumanMessage

In [53]:
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is. """

contextualize_q_system_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}"),
])

In [54]:
history_aware_retriever = create_history_aware_retriever(llm,retriever,contextualize_q_system_prompt)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x128d8e270>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChu

In [56]:
## Create a new document chain with history
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Context: {context}"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human","{input}")
])
question_answere_chain = create_stuff_documents_chain(llm,qa_prompt)

##Create Conversational RAG Chain
converesational_rag_chain = create_retrieval_chain(history_aware_retriever, question_answere_chain)
print("Converastaional RAG Chain Created!")

Converastaional RAG Chain Created!


In [58]:
chat_history=[]
#First Question
result1 = converesational_rag_chain.invoke({
    "chat_history":chat_history,
    "input":"What is Machine Learning?"
})
print(f"Q:What is machine learning?")
print(f"A: {result1['answer']}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q:What is machine learning?
A: Machine learning is a subset of artificial intelligence that allows systems to learn and improve from experience without explicit programming. It involves different techniques such as supervised learning (using labeled data), unsupervised learning (finding patterns in unlabeled data), and reinforcement learning (learning through interaction).


In [60]:
chat_history.extend([
    HumanMessage(content="what is machine Learning"),
    AIMessage(content = result1['answer'])
])

In [61]:
chat_history

[HumanMessage(content='what is machine Learning', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Machine learning is a subset of artificial intelligence that allows systems to learn and improve from experience without explicit programming. It involves different techniques such as supervised learning (using labeled data), unsupervised learning (finding patterns in unlabeled data), and reinforcement learning (learning through interaction).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [62]:
##Second Message
result2 = converesational_rag_chain.invoke({
    "chat_history":chat_history,
    "input":"What are its main types?"
})
result2

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{'chat_history': [HumanMessage(content='what is machine Learning', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Machine learning is a subset of artificial intelligence that allows systems to learn and improve from experience without explicit programming. It involves different techniques such as supervised learning (using labeled data), unsupervised learning (finding patterns in unlabeled data), and reinforcement learning (learning through interaction).', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'input': 'What are its main types?',
 'context': [Document(metadata={'source': 'data/doc_0.txt'}, page_content='Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learnin

In [63]:
result2['answer']

'The main types of machine learning include supervised learning, where models use labeled data to learn; unsupervised learning, which identifies patterns in unlabeled data; and reinforcement learning, where agents learn by interacting with their environment.'

## Using GROQ LLM's

In [64]:
llm

ChatHuggingFace(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.2', 'langchain-huggingface': '1.2.3'}}, llm=HuggingFacePipeline(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.2', 'langchain-huggingface': '1.2.3'}}, pipeline=TextGenerationPipeline: {'model': 'Qwen2ForCausalLM', 'dtype': 'bfloat16', 'device': 'mps', 'input_modalities': 'text', 'output_modalities': ('text',)}, model_id='Qwen/Qwen2.5-1.5B-Instruct', model_kwargs={'dtype': 'auto'}, pipeline_kwargs={'max_new_tokens': 256, 'do_sample': False, 'return_full_text': False}), tokenizer=Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-1.5B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstri

In [67]:
load_dotenv()

True

In [ ]:
import os
# os.getenv("GROQ_API_KEY")

In [69]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [70]:
os.environ['GROQ_API_KEY'] = os.getenv("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(model = "gemma2-9b-it")


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.10', 'langchain': '1.4.2'}}, profile={'name': 'Gemma 2 9B', 'status': 'deprecated', 'release_date': '2024-06-27', 'last_updated': '2024-06-27', 'open_weights': True, 'max_input_tokens': 8192, 'max_output_tokens': 8192, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x15748ba10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x157678590>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))